# SignalUp AI — Run Notebook

## Purpose

Use this notebook after training the complete ML notebook.

This file loads:

```text
artifacts/models/signalup_upgrade_propensity_model.joblib
artifacts/metrics/model_metadata.json
signalup_ai_dataset.csv
```

Then it generates:

- Single account prediction
- Batch predictions
- Propensity category
- Recommended marketing action
- Exportable prediction CSV

In [8]:
#Optional install
!pip install pandas numpy scikit-learn joblib


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [13]:
DATA_PATH = "data/signalup_ai_dataset.csv"

MODEL_PATH = Path("artifacts/models/signalup_upgrade_propensity_model.joblib")
METADATA_PATH = Path("artifacts/metrics/model_metadata.json")
OUTPUT_DIR = Path("run_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("signalup_ai_dataset.csv not found. Keep it in the same folder as this notebook.")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "Model artifact not found. Run SignalUp_AI_Complete_ML_Project.ipynb first to create artifacts/models/signalup_upgrade_propensity_model.joblib"
    )

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        "Metadata file not found. Run SignalUp_AI_Complete_ML_Project.ipynb first."
    )

df = pd.read_csv(DATA_PATH)
pipeline = joblib.load(MODEL_PATH)

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

print("Dataset shape:", df.shape)
print("Loaded model:", metadata["best_model"])
print("Model path:", MODEL_PATH)

FileNotFoundError: Model artifact not found. Run SignalUp_AI_Complete_ML_Project.ipynb first to create artifacts/models/signalup_upgrade_propensity_model.joblib

In [ ]:
feature_cols = metadata["features_used"]
target_col = metadata["target"]

missing_features = [col for col in feature_cols if col not in df.columns]

if missing_features:
    raise ValueError(f"Dataset is missing required model features: {missing_features}")

X = df[feature_cols].copy()

print("Feature count:", len(feature_cols))
X.head()

In [ ]:
def propensity_category(score: float) -> str:
    if score >= 0.80:
        return "VERY_HIGH"
    if score >= 0.60:
        return "HIGH"
    if score >= 0.40:
        return "MEDIUM"
    if score >= 0.20:
        return "LOW"
    return "VERY_LOW"


def recommended_marketing_action(category: str) -> str:
    mapping = {
        "VERY_HIGH": "Immediate Premium Trial Offer",
        "HIGH": "Discount + Premium Feature Campaign",
        "MEDIUM": "Educational Nurture Campaign",
        "LOW": "Re-engagement Campaign",
        "VERY_LOW": "Wait and Monitor",
    }
    return mapping.get(category, "Wait and Monitor")


def predict_accounts(input_df: pd.DataFrame) -> pd.DataFrame:
    X_input = input_df[feature_cols].copy()
    probabilities = pipeline.predict_proba(X_input)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    output = input_df[["Account_ID"]].copy() if "Account_ID" in input_df.columns else pd.DataFrame(index=input_df.index)
    output["predicted_upgrade"] = predictions
    output["upgrade_probability"] = np.round(probabilities, 4)
    output["propensity_category"] = output["upgrade_probability"].apply(propensity_category)
    output["recommended_marketing_action"] = output["propensity_category"].apply(recommended_marketing_action)

    if target_col in input_df.columns:
        output["actual_upgrade"] = input_df[target_col].values

    return output

## Single Account Prediction

In [ ]:
ACCOUNT_ID = df["Account_ID"].iloc[0]

single_row = df[df["Account_ID"] == ACCOUNT_ID]

single_prediction = predict_accounts(single_row)

single_prediction

## Batch Prediction

In [ ]:
batch_predictions = predict_accounts(df)

batch_predictions.head(20)

In [ ]:
# Propensity category summary

summary = (
    batch_predictions["propensity_category"]
    .value_counts()
    .rename_axis("propensity_category")
    .reset_index(name="count")
)

summary["percentage"] = (summary["count"] / len(batch_predictions) * 100).round(2)

summary

In [ ]:
# High priority upgrade candidates

high_priority = batch_predictions[
    batch_predictions["propensity_category"].isin(["VERY_HIGH", "HIGH"])
].sort_values("upgrade_probability", ascending=False)

high_priority.head(50)

In [ ]:
# Export prediction outputs

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

all_predictions_path = OUTPUT_DIR / f"signalup_predictions_{timestamp}.csv"
high_priority_path = OUTPUT_DIR / f"signalup_high_priority_upgrade_candidates_{timestamp}.csv"

batch_predictions.to_csv(all_predictions_path, index=False)
high_priority.to_csv(high_priority_path, index=False)

print("All predictions saved:", all_predictions_path)
print("High priority candidates saved:", high_priority_path)

## FastAPI Integration Helper

Use this logic inside your backend prediction service after loading the saved model artifact.

In [ ]:
def backend_style_prediction_response(account_id: str) -> dict:
    row = df[df["Account_ID"] == account_id]

    if row.empty:
        raise ValueError(f"Account_ID not found: {account_id}")

    prediction = predict_accounts(row).iloc[0].to_dict()

    return {
        "account_id": prediction["Account_ID"],
        "prediction_score": float(prediction["upgrade_probability"]),
        "predicted_upgrade": int(prediction["predicted_upgrade"]),
        "propensity_category": prediction["propensity_category"],
        "recommended_marketing_action": prediction["recommended_marketing_action"],
        "model_name": metadata["best_model"],
        "model_version": "v1.0.0",
    }


backend_style_prediction_response(ACCOUNT_ID)